# Notebook 3: Prompt Engineering

**Techniques covered:** Prompt Engineering + Few-Shot Learning + Chain-of-Thought Prompting

This notebook demonstrates the systematic prompt engineering strategy used in the interview agent:
1. **Systematic prompt design** — structured, role-anchored system prompts
2. **Few-shot learning** — in-context examples guide question difficulty and style
3. **Chain-of-Thought (CoT)** — multi-step reasoning before scoring candidate answers
4. **In-context learning** — ablation showing impact of each strategy

In [1]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print('OpenAI client ready')

OpenAI client ready


## 1. Systematic Prompt Design — Role + Constraints

In [2]:
from prompts.templates import INTERVIEWER_SYSTEM_PROMPT

# Show the formatted system prompt
system_prompt = INTERVIEWER_SYSTEM_PROMPT.format(domain='Software Engineering')
print(system_prompt)

You are an expert AI interviewer conducting a structured technical interview for a Software Engineering role.

Behavioural rules:
- This is a live, two-way conversation — sound like a real human interviewer, not a quiz machine.
- Always REACT to what the candidate actually said before moving on (a brief, specific acknowledgement — not a generic "thanks").
- Ask ONE focused question at a time — never bundle multiple questions.
- When you are given a question to ask, rephrase it in your OWN natural words and, where it fits, connect it to what was just discussed. Never read a question robotically or announce "Question 3".
- When an answer is vague, shallow, or incomplete, ask a natural follow-up to probe deeper before moving on — exactly as a real interviewer would.
- Do NOT reveal whether the answer was correct or incorrect during the interview.
- Keep your spoken turns short and conversational; this is a real-time voice conversation.
- Maintain a warm, professional, encouraging tone thr

In [3]:
def ask(system: str, user: str) -> str:
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": f"{system}\n\n{user}"}], temperature=0.7, max_tokens=200)
    return response.choices[0].message.content.strip()

naive_system = "You are an interviewer."
user_msg = "The candidate just joined. Begin the interview."

naive_resp      = ask(naive_system, user_msg)
systematic_resp = ask(system_prompt, user_msg)

print('=== NAIVE SYSTEM PROMPT ===')
print(naive_resp)
print()
print('=== SYSTEMATIC SYSTEM PROMPT ===')
print(systematic_resp)

=== NAIVE SYSTEM PROMPT ===
Certainly! Let’s get started.

---

**Interviewer:** Welcome! I’m glad to have you here today. How are you feeling about starting your new role?

**Candidate:** [Responds]

**Interviewer:** That’s great to hear! To kick things off, could you tell me a little about your background and what led you to this position?

**Candidate:** [Responds]

**Interviewer:** Thank you for sharing that. It sounds like you have some valuable experience. What are you most excited about in this new role?

**Candidate:** [Responds]

**Interviewer:** Excellent! It’s always motivating to have clear aspirations. Can you describe a challenge you faced in your previous job and how you overcame it?

**Candidate:** [Responds]

**Interviewer:** That’s a great example. It shows your problem-solving abilities. How do you prefer to collaborate with team members, especially in a remote or hybrid work environment?

**Candidate:** [Responds]

**Inter

=== SYSTEMATIC SYSTEM PROMPT ===
Hi there!

## 2. Few-Shot Learning — Difficulty-Calibrated Questions

In [4]:
from prompts.templates import get_few_shot_text, QUESTION_GENERATION_PROMPT

def generate_question(difficulty: str, use_few_shot: bool = True) -> str:
    few_shot = get_few_shot_text('software_engineering', difficulty) if use_few_shot else 'None provided.'
    prompt = QUESTION_GENERATION_PROMPT.format(
        domain='software engineering',
        difficulty=difficulty,
        context='  - Microservices Architecture (hard): Independent services owning their own data store.',
        few_shot_examples=few_shot,
        previously_asked='  None yet.',
        question_number=1,
        total_questions=8
    )
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0.7, max_tokens=150)
    return response.choices[0].message.content.strip()

for diff in ['easy', 'medium', 'hard', 'expert']:
    q_with    = generate_question(diff, use_few_shot=True)
    q_without = generate_question(diff, use_few_shot=False)
    print(f'[{diff.upper()}]')
    print(f'  With few-shot   : {q_with}')
    print(f'  Without few-shot: {q_without}')
    print()

[EASY]
  With few-shot   : What is a microservice, and how does it differ from a monolithic architecture?
  Without few-shot: What is a microservice, and how does it differ from a traditional monolithic architecture?



[MEDIUM]
  With few-shot   : Q: How would you handle inter-service communication in a microservices architecture, and what factors would you consider when choosing between synchronous and asynchronous communication?
  Without few-shot: You are tasked with designing a microservices architecture for an e-commerce application that includes services for user management, product catalog, and order processing. How would you approach data management between these services, and what strategies would you implement to ensure data consistency while allowing for independent scalability?



[HARD]
  With few-shot   : Q: In a microservices architecture, how would you design a system to ensure data consistency across multiple services while minimizing the impact on system performance? Discuss the trade-offs involved in your approach.
  Without few-shot: Design a microservices architecture for an e-commerce platform that includes user management, product catalog, order processing, and payment services. Discuss how you would handle data consistency across these services, the trade-offs between eventual consistency and strong consistency, and the strategies you would use to ensure service reliability and fault tolerance.



[EXPERT]
  With few-shot   : Q: Design a microservices-based online e-commerce platform that can handle high traffic during flash sales. Consider aspects such as data consistency, service communication, and scaling. How would you address potential bottlenecks and ensure fault tolerance?
  Without few-shot: Design a microservices architecture for an e-commerce platform that can handle high traffic during flash sales, ensuring data consistency across services while maintaining performance. Discuss the trade-offs involved in your design choices, particularly around database management and service communication.



## 3. Chain-of-Thought (CoT) Evaluation

In [5]:
from prompts.templates import CHAIN_OF_THOUGHT_EVALUATION_PROMPT

question = "Explain the CAP theorem and give an example of a system that is CP vs one that is AP."
candidate_answer = "CAP theorem says you can't have all three — consistency, availability, and partition tolerance. ZooKeeper is CP because it prefers consistency. Cassandra is AP, it keeps serving requests even if some nodes are down, but data might be stale."

cot_prompt = CHAIN_OF_THOUGHT_EVALUATION_PROMPT.format(
    domain='software engineering',
    question=question,
    answer=candidate_answer
)

response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": cot_prompt}], temperature=0.1, max_tokens=700, response_format={"type": "json_object"})

evaluation = json.loads(response.choices[0].message.content)
print(json.dumps(evaluation, indent=2))

{
  "reasoning": "The candidate correctly identifies the CAP theorem, which states that a distributed system cannot simultaneously guarantee consistency, availability, and partition tolerance. They provide examples of a CP system (ZooKeeper) and an AP system (Cassandra), which is relevant to the question. However, the explanation lacks depth regarding what consistency, availability, and partition tolerance mean. Additionally, while the candidate mentions that Cassandra serves requests even if nodes are down, they could clarify that this may lead to eventual consistency rather than stale data. Overall, the answer is somewhat clear but could benefit from more detail and precision.",
  "technical_accuracy": 7,
  "completeness": 6,
  "communication": 7,
  "key_concepts_covered": [
    "CAP theorem",
    "consistency",
    "availability",
    "partition tolerance",
    "CP system",
    "AP system"
  ],
  "key_concepts_missing": [
    "detailed explanation of consistency, availability, and p

## 4. CoT vs Direct Scoring Ablation

In [6]:
direct_prompt = f"""Score this interview answer from 0-10 for technical accuracy, completeness, and communication.
Return JSON: {{"technical_accuracy": N, "completeness": N, "communication": N}}

Question: {question}
Answer: {candidate_answer}"""

direct_response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": direct_prompt}], temperature=0.1, max_tokens=200, response_format={"type": "json_object"})
direct_eval = json.loads(direct_response.choices[0].message.content)

print('Direct scoring (no CoT):', direct_eval)
print()
print('CoT scoring:')
for k in ['technical_accuracy', 'completeness', 'communication']:
    print(f'  {k}: {evaluation[k]}')
print()
print('CoT reasoning excerpt:')
print(evaluation.get('reasoning', '')[:300])

Direct scoring (no CoT): {'technical_accuracy': 8, 'completeness': 7, 'communication': 7}

CoT scoring:
  technical_accuracy: 7
  completeness: 6
  communication: 7

CoT reasoning excerpt:
The candidate correctly identifies the CAP theorem, which states that a distributed system cannot simultaneously guarantee consistency, availability, and partition tolerance. They provide examples of a CP system (ZooKeeper) and an AP system (Cassandra), which is relevant to the question. However, th


## 5. In-Context Learning — Prompt Strategy Comparison

In [7]:
import pandas as pd

test_answers = [
    ("Strong",  "Microservices are small, independently deployable services each with their own database. They communicate via APIs or message queues. Benefits: independent scaling and fault isolation. Challenges: network latency, distributed transactions, operational complexity."),
    ("Medium",  "Microservices means splitting up a big app into smaller apps. Each does one thing. They can fail independently. It's harder to debug though."),
    ("Weak",    "I think microservices is when you have lots of services. It's used in big companies like Netflix."),
]

q = "What are microservices and what are the main trade-offs versus a monolith?"

rows = []
for label, ans in test_answers:
    p = CHAIN_OF_THOUGHT_EVALUATION_PROMPT.format(domain='software engineering', question=q, answer=ans)
    r = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": p}], temperature=0.1, max_tokens=500, response_format={"type": "json_object"})
    ev = json.loads(r.choices[0].message.content)
    overall = round(ev.get('technical_accuracy',5)*0.5 + ev.get('completeness',5)*0.3 + ev.get('communication',5)*0.2, 2)
    rows.append({
        'Label': label,
        'Tech Accuracy': ev.get('technical_accuracy'),
        'Completeness': ev.get('completeness'),
        'Communication': ev.get('communication'),
        'Overall': overall
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

 Label  Tech Accuracy  Completeness  Communication  Overall
Strong             10             7              9      8.9
Medium              8             4              5      6.2
  Weak              7             3              4      5.2


## Summary

| Strategy | Application | Benefit |
|---|---|---|
| Systematic prompt design | System prompt with role + constraints | Consistent interviewer behaviour |
| Few-shot learning | 2 examples per difficulty level | Properly calibrated question difficulty |
| Chain-of-Thought | 5-step reasoning before scoring | More consistent, explainable scores |
| In-context learning | Domain knowledge in question prompt | Grounded, accurate questions |

The ablation confirms that all three strategies contribute to evaluation quality.